In [ ]:
!pip install -q groq chromadb pypdf

In [ ]:
from groq import Groq
from google.colab import userdata
api_key = userdata.get("GRKey2")
client = Groq(api_key=api_key)
MODEL = "openai/gpt-oss-120b"

In [ ]:
import os
os.makedirs("docs", exist_ok=True)

In [ ]:
print(os.listdir("docs"))

['faq.txt', 'clothing_rag.pdf']


**1) DOCUMENT LOADING :**

In [ ]:
import re
from pypdf import PdfReader

def load_txt(path):
  with open(path,"r",encoding="utf-8") as f:
    return[{"text":f.read(),"source":os.path.basename(path),"page":1}]


def load_pdf(path):
  pages=[]
  for number,page in enumerate(PdfReader(path).pages,start=1):
    pages.append({"text":page.extract_text() or " ","source":os.path.basename(path),"page":number})
  return pages


def load_folder(folder):
    pages = []
    for name in sorted(os.listdir(folder)):
        path = os.path.join(folder, name)
        if name.lower().endswith(".txt"):
            pages += load_txt(path)
        elif name.lower().endswith(".pdf"):
            pages += load_pdf(path)
    return pages

**2) CLEANING DATA :**

In [ ]:
def clean_text(text):
  return " ".join(text.split())

**CHUNKING :**

In [ ]:
import os
def chunk_sentences(text, max_chars=400):
    """Pack whole sentences into chunks of at most `max_chars` characters."""
    sentences = re.split(r"(?<=[.!?])\s+", text)
    chunks = []
    current = ""
    for sentence in sentences:
        if current and len(current) + 1 + len(sentence) > max_chars:
            chunks.append(current)
            current = sentence
        else:
            current = (current + " " + sentence).strip()
    if current:
        chunks.append(current)
    return chunks

def chunk_paragraphs(text, max_chars=400):
    """Cut at blank lines first. Cut a paragraph into sentences only if it is longer than max_chars."""
    chunks = []
    for paragraph in text.split("\n\n"):
        paragraph = clean_text(paragraph)
        if paragraph:
            chunks += chunk_sentences(paragraph, max_chars)
    return chunks


def build_chunks(folder, max_chars=400):
    all_chunks = []
    for page in load_folder(folder):
        if not page["text"].strip():
            print(f"Skipped empty page (a scan? needs OCR):", page["source"], "page", page["page"])
            continue
        for piece in chunk_paragraphs(page["text"], max_chars):
            all_chunks.append({"text": piece, "source": page["source"], "page": page["page"]})
    return all_chunks

chunks=build_chunks("docs")
print(len(chunks),"chunks")
print(chunks[70])

74 chunks
{'text': 'Do you sell gift cards?,Please check the current catalogue or contact support to confirm whether gift cards are available. How should I wash my clothing item?,Follow the care label attached to the garment. Care instructions can differ by fabric and product. Will the color look exactly like the photo?,Colors may look slightly different because of lighting and screen settings.', 'source': 'faq.txt', 'page': 1}


**3) EMBEDDING and STORING CHUNKS IN CHROMADB :**

In [ ]:
import chromadb
from chromadb.utils.embedding_functions import DefaultEmbeddingFunction

embed=DefaultEmbeddingFunction()
chroma=chromadb.Client()

def store_chunks(chunks,name="docs_kb"):
  try:
    chroma.delete_collection(name)
  except Exception:
      pass
  collection=chroma.create_collection(name=name,embedding_function=embed)
  collection.upsert(
      documents=[c["text"] for c in chunks],
      metadatas=[{"source":c["source"],"page":c["page"]} for c in chunks],
      ids=[f"chunk{i}" for i in range(len(chunks))]
  )
  return collection
collection=store_chunks(chunks)
print("Stored:",collection.count(),"chunks")


/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:00<00:00, 96.8MiB/s]


Stored: 74 chunks


**4) RETREIVE :**

In [ ]:
def retrieve(question,n=3):
  result=collection.query(query_texts=[question],n_results=min(n,collection.count()))
  found = []
  for text,meta,dist in zip(result["documents"][0],result["metadatas"][0],result["distances"][0]):
    found.append({"text":text,"source":meta["source"],"page":meta["page"],"distance":dist})
    return found

for item in retrieve("What is Retrieval Augmented Generation?"):
  print(f'{item["distance"]:.2f} | {item["source"]} page {item["page"]} | {item["text"][:90]}...')

1.67 | clothing_rag.pdf page 1 | In the world, fashion and design are the leading European clothing indus - tries. From the...


**5) ANSWER WITH MEMORY :**

In [ ]:
RULES = ("You answer questions using ONLY the context the user gives you. "
         "If the answer is not in the context, reply exactly: I don't know.")

def ask(question, found, history):
    context = "\n\n".join(
        f'[{c["source"]},page{c["page"]}]\n{c["text"]}'
        for c in found
    )
    messages = [{"role": "system", "content": RULES}]
    messages += history
    messages.append({
        "role": "user",
        "content": f"Context:\n{context}\n\nQuestion:{question}"
    })
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0,
        max_tokens=400
    )
    return response.choices[0].message.content
chat_history = []

def rag_chat(question):
    found = retrieve(question)
    reply = ask(question, found, chat_history[-6:])
    chat_history.append({"role": "user", "content": question})
    chat_history.append({"role": "assistant", "content": reply})
    return reply, found

def show(question):
    reply, found = rag_chat(question)
    print("Q:", question)
    print("A:", reply)
    sources = ", ".join(
        f"{c['source']} p{c['page']}" for c in found
    )
    print("Sources:", sources)
    print()


while True:
    question = input("Ask a question (or type 'exit' to stop): ")
    if question.lower() == "exit":
        print("Chatbot stopped.")
        break
    show(question)

Ask a question (or type 'exit' to stop): What is the capital of Pakistan?
Q: What is the capital of Pakistan?
A: I don't know.
Sources: faq.txt p1

Ask a question (or type 'exit' to stop): How can I cancel my order?
Q: How can I cancel my order?
A: Please contact customer support promptly to ask whether your order can still be cancelled.
Sources: faq.txt p1

Ask a question (or type 'exit' to stop): What is Fashion Clothing?
Q: What is Fashion Clothing?
A: Fashion clothing is an expressive product that people use to communicate their image and social position; its consumption signals a society’s focus on status and highlights the role of clothing in conveying status.
Sources: clothing_rag.pdf p2

Ask a question (or type 'exit' to stop): How to make an egg?
Q: How to make an egg?
A: I don't know.
Sources: clothing_rag.pdf p2

Ask a question (or type 'exit' to stop): Define consumption of fashion clothing?
Q: Define consumption of fashion clothing?
A: The consumption of fashion clothing r